# Phase A (Notebook 3): Two-Pass VLM Scene Captioning & Knowledge Bundle Assembly (`PhaseA3_VLM_Captioning.ipynb`)

**Environment Stack**: Clean `Qwen/Qwen2.5-VL-7B-Instruct` environment (`transformers>=4.49.0`, `accelerate`, `qwen-vl-utils` on Colab GPU L4 24GB / A100) isolated from Visual and Audio dependencies.

### Workflow Overview
1. **Cell 1 — Hardware Check, Google Drive Mount & Input Verification**:
   - Verifies GPU (`L4` / `A100`), mounts Drive, and checks `shots_and_scenes.json`, `attributed_subtitles.json`, and `keyframes/`.
2. **Cell 2 — Clone / Pull Latest Code from GitHub (`/content/MovieRag_repo`)**:
   - Clones/pulls `main`, including `char_bank/sintel/external_plot.txt` and `pipeline/stage5_vlm_caption.py`.
3. **Cell 3 — Install Clean VLM Dependencies**:
   - Installs latest `transformers>=4.49.0`, `accelerate`, and `qwen-vl-utils` without downgrading GPU PyTorch.
4. **Cell 4 — Configure Local SSD HuggingFace Cache**:
   - Keeps `Qwen2.5-VL-7B-Instruct` (~15 GB) on fast local NVMe SSD (`/content/local_models_cache/huggingface`) so it does not exhaust Google Drive's 15 GB storage quota.
5. **Cell 5 (Optional 1-Scene Diagnostic) — Inspect Scene 4 Output & Memory**:
   - Tests Pass 1 (`vlm_caption`) + Pass 2 (`exit_state`) on `sintel_scene_0004` (the Shaman hut scene) and prints runtime + RAM/VRAM before captioning all 34 scenes.
6. **Cell 6 — Stage A5 (`stage5_vlm_caption.py`) Full Run & Bundle Assembly**:
   - Runs Pass 1 + Pass 2 across all 34 scenes (resumable via `vlm_captions.jsonl`), subsampling $\le 8$ keyframes/scene, trimming host & GPU memory after every scene, and writing `movie_knowledge_bundle.json`.
7. **Cell 7 — Knowledge Bundle Verification & Inspection**:
   - Validates schema completeness, `external_plot_documents`, subtitles, `vlm_caption`, and `exit_state` across sample scenes.

In [ ]:
# ==============================================================================
# CELL 1: GPU Check, Google Drive Mount & Stage A1-A4 Artifact Check
# ==============================================================================
import os
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running inside Google Colab; using local paths.")

MOVIE_ID = "sintel"                              # @param {type:"string"}
MOVIE_TITLE = "Sintel"                           # @param {type:"string"}
FRANCHISE_ID = "sintel"                          # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MovieRAG"   # @param {type:"string"}
VLM_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"     # @param {type:"string"}
MAX_KEYFRAMES_PER_SCENE = 8                      # @param {type:"integer"}

CHAR_BANK_DIR = f"{DRIVE_ROOT}/char_bank/{MOVIE_ID}"
WORKDIR = f"{DRIVE_ROOT}/workdir/{MOVIE_ID}"
LOCAL_HF_CACHE = "/content/local_models_cache/huggingface"
os.makedirs(LOCAL_HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = LOCAL_HF_CACHE
os.environ["HF_HUB_CACHE"] = f"{LOCAL_HF_CACHE}/hub"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu_info.stdout.strip().splitlines()[0] if gpu_info.stdout.strip() else "NO GPU DETECTED")

for label, path in [
    ("shots_and_scenes.json", f"{WORKDIR}/shots_and_scenes.json"),
    ("attributed_subtitles.json", f"{WORKDIR}/attributed_subtitles.json"),
    ("keyframes dir", f"{WORKDIR}/keyframes"),
    ("character_glossary.json", f"{CHAR_BANK_DIR}/character_glossary.json"),
]:
    ok = os.path.exists(path)
    print(f"  [{'OK' if ok else 'MISSING'}] {label}: {path}")

In [ ]:
# ==============================================================================
# CELL 2: Clone / Pull Latest Source Code from GitHub (/content/MovieRag_repo)
# ==============================================================================
import importlib
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/ctu-flix/MovieRag.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"                                        # @param {type:"string"}
REPO_SUBDIR = ""
CLONE_DIR = "/content/MovieRag_repo"

github_token = None
if globals().get("IN_COLAB", False):
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

auth_repo_url = GITHUB_REPO_URL
if github_token and GITHUB_REPO_URL.startswith("https://github.com/"):
    auth_repo_url = GITHUB_REPO_URL.replace("https://github.com/", f"https://{github_token}@github.com/")

if not os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"Cloning {GITHUB_REPO_URL} (branch: {GITHUB_BRANCH})...")
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, auth_repo_url, CLONE_DIR], check=True)
else:
    print(f"Pulling latest commits from origin/{GITHUB_BRANCH}...")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", auth_repo_url], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)

if REPO_SUBDIR and os.path.isdir(os.path.join(CLONE_DIR, REPO_SUBDIR)):
    PROJECT_ROOT = os.path.join(CLONE_DIR, REPO_SUBDIR)
elif os.path.isdir(os.path.join(CLONE_DIR, "MovieRag", "pipeline")):
    PROJECT_ROOT = os.path.join(CLONE_DIR, "MovieRag")
else:
    PROJECT_ROOT = CLONE_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

for mod_name in [k for k in list(sys.modules) if k == "pipeline" or k.startswith("pipeline.")]:
    importlib.reload(sys.modules[mod_name])

commit_log = subprocess.run(["git", "-C", CLONE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"Active PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Latest Commit      : {commit_log}")

In [ ]:
# ==============================================================================
# CELL 3: Install Clean Qwen2.5-VL Dependencies
# ==============================================================================
import subprocess
import sys

pkgs = [
    "transformers>=4.49.0",
    "accelerate>=0.34.0",
    "qwen-vl-utils",
    "pillow",
    "psutil",
]
print("Installing VLM dependencies:", ", ".join(pkgs))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"] + pkgs, check=True)

import torch
import transformers
print(f"torch={torch.__version__} (CUDA={torch.cuda.is_available()}, bf16={torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False})")
print(f"transformers={transformers.__version__}")

In [ ]:
# ==============================================================================
# CELL 4: Verify Local SSD Disk Space & Apply Manual Subtitle Speaker Overrides
# ==============================================================================
import os
import shutil
import sys

_disk = shutil.disk_usage("/content" if os.path.isdir("/content") else ".")
print(f"Local disk free space: {_disk.free / (1024**3):.1f} GB")
print("Note: Qwen/Qwen2.5-VL-7B-Instruct (~15 GB) downloads directly to Local SSD HF_HOME")
print("to keep your Google Drive storage quota clean while persisting JSON outputs on Drive.")

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/MovieRag_repo" if os.path.isdir("/content/MovieRag_repo/pipeline") else "/content/MovieRag_repo/MovieRag"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.stage4_subtitles import apply_subtitle_speaker_fixes

_movie_id = globals().get("MOVIE_ID", "sintel")
_drive_root = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/MovieRAG")
_workdir = globals().get("WORKDIR", f"{_drive_root}/workdir/{_movie_id}")

# Pure manual overrides keyed by utt_id (e.g. "utt_0009"), 2-decimal start_time ("143.22"), or exact text:
SPEAKER_OVERRIDES = {
    "utt_0009": "Sintel",  # [143.22s - 144.16s] A dragon.
    "utt_0011": "Sintel",  # [152.82s - 155.10s] I've been single since I've known.
    "utt_0012": "Sintel",  # [206.92s - 209.33s] I'm almost done...
    "utt_0013": "Sintel",  # [210.47s - 211.88s] Be good !
    "utt_0014": "Sintel",  # [227.75s - 229.24s] Good night, Scales.
    "utt_0019": "Sintel",  # [445.58s - 446.64s] I didn't succeed.
    "utt_0025": "Sintel",  # [625.56s - 627.11s] Scales...
}

fix_summary = apply_subtitle_speaker_fixes(
    output_dir=_workdir,
    speaker_overrides=SPEAKER_OVERRIDES,
)
print(f"\nApplied {fix_summary['updated_count']} manual override(s) in attributed_subtitles.json & shots_and_scenes.json")
print(f"Speaker distribution: {fix_summary['speaker_counts']}")
print("\n--- SUBTITLES GOING INTO VLM PROMPT ---")
for i, u in enumerate(fix_summary["utterances"]):
    uid = u.get("utt_id", f"utt_{i:04d}")
    print(f"  {uid} [{u['start_time']:6.2f}s - {u['end_time']:6.2f}s] {u['speaker']:8s} ({u['source']}): {u['text']}")

In [ ]:
# ==============================================================================
# CELL 5 (OPTIONAL DIAGNOSTIC): Test 1 Scene Before Running Full Movie
# ==============================================================================
import os
import sys

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/MovieRag_repo" if os.path.isdir("/content/MovieRag_repo/pipeline") else "/content/MovieRag_repo/MovieRag"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.stage5_vlm_caption import run_single_scene_diagnostic

_movie_id = globals().get("MOVIE_ID", "sintel")
_movie_title = globals().get("MOVIE_TITLE", "Sintel")
_drive_root = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/MovieRAG")
_workdir = globals().get("WORKDIR", f"{_drive_root}/workdir/{_movie_id}")
_char_bank = globals().get("CHAR_BANK_DIR", f"{_drive_root}/char_bank/{_movie_id}")
_vlm_model = globals().get("VLM_MODEL_ID", "Qwen/Qwen2.5-VL-7B-Instruct")
_max_kfs = globals().get("MAX_KEYFRAMES_PER_SCENE", 8)

DIAGNOSTIC_SCENE_INDEX = 4  # sintel_scene_0004 (Shaman hut scene)

_diag = run_single_scene_diagnostic(
    output_dir=_workdir,
    scene_index=DIAGNOSTIC_SCENE_INDEX,
    movie_title=_movie_title,
    char_bank_dir=_char_bank,
    vlm_model_id=_vlm_model,
    max_keyframes_per_scene=_max_kfs,
)

In [ ]:
# ==============================================================================
# CELL 6: Stage A5 - Full 2-Pass Qwen2.5-VL Captioning & Bundle Assembly
# ==============================================================================
import os
import sys

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/MovieRag_repo" if os.path.isdir("/content/MovieRag_repo/pipeline") else "/content/MovieRag_repo/MovieRag"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.stage5_vlm_caption import run_stage5_vlm_caption

_movie_id = globals().get("MOVIE_ID", "sintel")
_movie_title = globals().get("MOVIE_TITLE", "Sintel")
_franchise_id = globals().get("FRANCHISE_ID", _movie_id)
_drive_root = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/MovieRAG")
_workdir = globals().get("WORKDIR", f"{_drive_root}/workdir/{_movie_id}")
_char_bank = globals().get("CHAR_BANK_DIR", f"{_drive_root}/char_bank/{_movie_id}")
_vlm_model = globals().get("VLM_MODEL_ID", "Qwen/Qwen2.5-VL-7B-Instruct")
_max_kfs = globals().get("MAX_KEYFRAMES_PER_SCENE", 8)
RESTART_STAGE5 = False  # Set True only if you want to regenerate all captions from scratch

stage5_summary = run_stage5_vlm_caption(
    output_dir=_workdir,
    movie_title=_movie_title,
    franchise_id=_franchise_id,
    char_bank_dir=_char_bank,
    vlm_model_id=_vlm_model,
    max_keyframes_per_scene=_max_kfs,
    restart=RESTART_STAGE5,
)

print("\nStage A5 Summary:")
for k, v in stage5_summary.items():
    print(f"  {k}: {v}")

In [ ]:
# ==============================================================================
# CELL 7: Verify Final movie_knowledge_bundle.json Schema & Content
# ==============================================================================
import json
from pathlib import Path

_movie_id = globals().get("MOVIE_ID", "sintel")
_workdir = Path(globals().get("WORKDIR", f"/content/drive/MyDrive/MovieRAG/workdir/{_movie_id}"))
bundle_path = _workdir / "movie_knowledge_bundle.json"

with open(bundle_path, "r", encoding="utf-8") as f:
    bundle = json.load(f)

scenes = bundle.get("scenes", [])
print("=== PHASE A FINAL BUNDLE VERIFICATION ===")
print(f"  Bundle File           : {bundle_path} ({bundle_path.stat().st_size / (1024**2):.2f} MB)")
print(f"  Movie ID / Title      : {bundle.get('movie_id')} / {bundle.get('movie_title')}")
print(f"  Duration / FPS        : {bundle.get('duration_sec')}s @ {bundle.get('fps')} fps")
print(f"  Glossary Characters   : {list(bundle.get('character_glossary', {}).keys())}")
print(f"  External Plot Docs    : {len(bundle.get('external_plot_documents', []))} (tiers: {[d.get('tier') for d in bundle.get('external_plot_documents', [])]})")
print(f"  Total Scenes          : {len(scenes)}")
print(f"  Scenes w/ vlm_caption : {sum(1 for s in scenes if s.get('vlm_caption'))} / {len(scenes)}")
print(f"  Scenes w/ exit_state  : {sum(1 for s in scenes if s.get('exit_state'))} / {len(scenes)}")
print(f"  Scenes w/ subtitles   : {sum(1 for s in scenes if s.get('subtitles'))} / {len(scenes)}")

for sc in [scenes[1], scenes[4], scenes[-2]] if len(scenes) >= 6 else scenes[:3]:
    print(f"\n--- {sc['scene_id']} [{sc['start_time']:.1f}s - {sc['end_time']:.1f}s] ---")
    print(f"  Characters : {[c['name'] for c in sc.get('characters_present', [])]}")
    print(f"  Subtitles  : {[(u['speaker'], u['text']) for u in sc.get('subtitles', [])]}")
    print(f"  Caption    : {sc.get('vlm_caption', '')[:320]}...")
    print(f"  Exit State : {sc.get('exit_state', '')}")